In [14]:
import pandas as pd

flight_data = pd.read_csv("airline_cleaned.csv")
weather_data = pd.read_csv("weather.csv")

In [15]:
print(flight_data.head())
print(weather_data.head())

   Quarter  Month  DayofMonth  DayOfWeek  CRSDepTime  CRSArrTime  Distance  \
0        1      1           1          3         659        1020    2475.0   
1        1      1           2          4         659        1020    2475.0   
2        1      1           3          5         659        1020    2475.0   
3        1      1           4          6         700        1026    2475.0   
4        1      1           5          7         659        1020    2475.0   

  Reporting_Airline Origin Dest  ArrDelayMinutes  
0                AA    JFK  LAX              0.0  
1                AA    JFK  LAX              2.0  
2                AA    JFK  LAX              0.0  
3                AA    JFK  LAX              0.0  
4                AA    JFK  LAX              0.0  
                  date  tavg  tmin  tmax  prcp  snow  wdir  wspd  wpgt  \
0  2024-05-15 00:00:00  23.9  19.0  28.0   0.0   NaN   NaN   7.8   NaN   
1  2024-05-16 00:00:00  23.1  20.0  27.0   0.3   NaN   NaN   5.7   NaN   
2  

In [16]:
flight_data['YEAR'] = 2024

In [17]:
print(flight_data.columns.tolist())

['Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'CRSDepTime', 'CRSArrTime', 'Distance', 'Reporting_Airline', 'Origin', 'Dest', 'ArrDelayMinutes', 'YEAR']


In [19]:
flight_data['date'] = pd.to_datetime(
    flight_data[['YEAR','Month','DayofMonth']]
        .rename(columns={
            'YEAR': 'year',
            'Month': 'month',
            'DayofMonth': 'day'
        })
)

In [20]:
weather_data['date'] = pd.to_datetime(weather_data['date'])

In [21]:
merged_data = pd.merge(
    flight_data,
    weather_data,
    on='date',
    how='left'
)


In [23]:
print(weather_data.columns.tolist())

['date', 'tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun']


In [24]:
merged_data = pd.merge(
    flight_data,
    weather_data,
    on='date',
    how='left'
)

In [25]:
merged_data = pd.merge(
    flight_data,
    weather_data,
    on='date',
    how='left'
)

print(merged_data.head())

   Quarter  Month  DayofMonth  DayOfWeek  CRSDepTime  CRSArrTime  Distance  \
0        1      1           1          3         659        1020    2475.0   
1        1      1           2          4         659        1020    2475.0   
2        1      1           3          5         659        1020    2475.0   
3        1      1           4          6         700        1026    2475.0   
4        1      1           5          7         659        1020    2475.0   

  Reporting_Airline Origin Dest  ...  tavg  tmin tmax  prcp  snow  wdir  wspd  \
0                AA    JFK  LAX  ...   NaN   NaN  NaN   NaN   NaN   NaN   NaN   
1                AA    JFK  LAX  ...   NaN   NaN  NaN   NaN   NaN   NaN   NaN   
2                AA    JFK  LAX  ...   NaN   NaN  NaN   NaN   NaN   NaN   NaN   
3                AA    JFK  LAX  ...   NaN   NaN  NaN   NaN   NaN   NaN   NaN   
4                AA    JFK  LAX  ...   NaN   NaN  NaN   NaN   NaN   NaN   NaN   

   wpgt  pres  tsun  
0   NaN   NaN   NaN  


In [26]:
print(merged_data.shape)
print(merged_data.columns)

(539747, 23)
Index(['Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'CRSDepTime',
       'CRSArrTime', 'Distance', 'Reporting_Airline', 'Origin', 'Dest',
       'ArrDelayMinutes', 'YEAR', 'date', 'tavg', 'tmin', 'tmax', 'prcp',
       'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun'],
      dtype='object')


In [27]:
print('ArrDelayMinutes' in merged_data.columns)

True


In [28]:
X = merged_data.drop(columns=['ArrDelayMinutes'])
y = merged_data['ArrDelayMinutes']

In [29]:
merged_data['delay_flag'] = (merged_data['ArrDelayMinutes'] > 15).astype(int)

X = merged_data.drop(columns=['ArrDelayMinutes','delay_flag'])
y = merged_data['delay_flag']

In [30]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [32]:
print(X_train.dtypes)

Quarter                       int64
Month                         int64
DayofMonth                    int64
DayOfWeek                     int64
CRSDepTime                    int64
CRSArrTime                    int64
Distance                    float64
Reporting_Airline            object
Origin                       object
Dest                         object
YEAR                          int64
date                 datetime64[ns]
tavg                        float64
tmin                        float64
tmax                        float64
prcp                        float64
snow                        float64
wdir                        float64
wspd                        float64
wpgt                        float64
pres                        float64
tsun                        float64
dtype: object


In [33]:
X = pd.get_dummies(X, drop_first=True)

In [38]:
merged_data = merged_data.dropna(subset=['ArrDelayMinutes'])

In [37]:
print(y.isna().sum())

17478


In [40]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# ==============================
# 1️⃣ Load your merged dataset
# ==============================
# If already loaded, skip this
# merged_data = pd.read_csv("your_merged_file.csv")

print("Original Shape:", merged_data.shape)

# ==============================
# 2️⃣ Remove rows where target is missing
# ==============================
merged_data = merged_data.dropna(subset=['ArrDelayMinutes'])

# ==============================
# 3️⃣ Take smaller sample (FASTER TRAINING)
# ==============================
merged_data = merged_data.sample(20000, random_state=42)

print("After Cleaning + Sampling:", merged_data.shape)

# ==============================
# 4️⃣ Define Target (y)
# ==============================
y = merged_data['ArrDelayMinutes']

# ==============================
# 5️⃣ Define Features (X)
# ==============================
X = merged_data.drop(columns=['ArrDelayMinutes'])

# Drop date if exists
if 'date' in X.columns:
    X = X.drop(columns=['date'])

# ==============================
# 6️⃣ Encode Categorical Columns
# ==============================
X = pd.get_dummies(X, drop_first=True)

print("Final Feature Shape:", X.shape)

# ==============================
# 7️⃣ Train Test Split
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ==============================
# 8️⃣ Train Model (FAST VERSION)
# ==============================
model = RandomForestRegressor(
    n_estimators=50,      # smaller = faster
    max_depth=10,         # limit depth
    random_state=42,
    n_jobs=-1             # use all CPU cores
)

model.fit(X_train, y_train)

# ==============================
# 9️⃣ Evaluate Model
# ==============================
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\nModel Performance:")
print("MAE:", round(mae, 2))
print("R2 Score:", round(r2, 3))

Original Shape: (522269, 24)
After Cleaning + Sampling: (20000, 24)
Final Feature Shape: (20000, 650)

Model Performance:
MAE: 11.7
R2 Score: 0.144


In [41]:
merged_data['date'] = pd.to_datetime(merged_data['date'])

merged_data['month'] = merged_data['date'].dt.month
merged_data['day'] = merged_data['date'].dt.day
merged_data['weekday'] = merged_data['date'].dt.weekday

In [42]:
merged_data['date'] = pd.to_datetime(merged_data['date'])

merged_data['month'] = merged_data['date'].dt.month
merged_data['day'] = merged_data['date'].dt.day
merged_data['weekday'] = merged_data['date'].dt.weekday

In [43]:
!pip install xgboost

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [44]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [45]:
merged_data['DelayCategory'] = (merged_data['ArrDelayMinutes'] > 15).astype(int)

In [47]:
!pip install shap
import shap


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached slicer-0.0.8-py3-none-any.whl.metadata (4.0 kB)
   ---------------------------------------- 0.0/549.3 kB ? eta -:--:--
   ------------------- -------------------- 262.1/549.3 kB ? eta -:--:--
   ---------------------------------------- 549.3/549.3 kB 3.7 MB/s eta 0:00:00
Using cached slicer-0.0.8-py3-none-any.whl (15 kB)
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   -------------------------------------- - 2.6/2.8 MB 12.5 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 10.0 MB/s eta 0:00:00
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
   ---------------------------------------- 0.0/38.1 MB ? eta -:--:--
   ---- ----------------------------------- 3.9/38.1 MB 19.5 MB/s eta 0:00:02
   --------- ------------------------------ 8.9/38.1 MB 20.5 MB/s eta 0:00:02
   -------------- ------

C:\Users\Admin\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [48]:
print(merged_data.columns)

Index(['Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'CRSDepTime',
       'CRSArrTime', 'Distance', 'Reporting_Airline', 'Origin', 'Dest',
       'ArrDelayMinutes', 'YEAR', 'date', 'tavg', 'tmin', 'tmax', 'prcp',
       'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun', 'delay_flag', 'month',
       'day', 'weekday', 'DelayCategory'],
      dtype='object')


In [49]:
merged_data = merged_data.drop(columns=[
    'Month',
    'DayofMonth',
    'DayOfWeek'
])

In [50]:
merged_data['hour'] = merged_data['CRSDepTime'] // 100

In [51]:
airline_avg = merged_data.groupby('Reporting_Airline')['ArrDelayMinutes'].mean()
merged_data['airline_avg_delay'] = merged_data['Reporting_Airline'].map(airline_avg)

In [52]:
train = merged_data[merged_data['date'] < '2025-09-01']
test = merged_data[merged_data['date'] >= '2025-09-01']

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)





In [54]:
print("Columns in your dataset:")
print(merged_data.columns.tolist())

Columns in your dataset:
['Quarter', 'CRSDepTime', 'CRSArrTime', 'Distance', 'Reporting_Airline', 'Origin', 'Dest', 'ArrDelayMinutes', 'YEAR', 'date', 'tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun', 'delay_flag', 'month', 'day', 'weekday', 'DelayCategory', 'hour', 'airline_avg_delay']


In [55]:
# List of possible leakage columns
possible_leakage = [
    'ArrDelay',
    'DepDelay',
    'ActualElapsedTime',
    'TaxiIn',
    'TaxiOut',
    'WheelsOn',
    'WheelsOff',
    'ArrTime'
]

# Check which ones exist in your data
existing_leakage = [col for col in possible_leakage if col in merged_data.columns]

print("Columns that may cause leakage:")
print(existing_leakage)

Columns that may cause leakage:
[]


In [56]:
# Features and target
y = merged_data['ArrDelayMinutes']
X = merged_data.drop(columns=['ArrDelayMinutes'])

# Optional: check columns you are using
print("Final features for training:")
print(X.columns.tolist())

Final features for training:
['Quarter', 'CRSDepTime', 'CRSArrTime', 'Distance', 'Reporting_Airline', 'Origin', 'Dest', 'YEAR', 'date', 'tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun', 'delay_flag', 'month', 'day', 'weekday', 'DelayCategory', 'hour', 'airline_avg_delay']


In [57]:
y = merged_data['ArrDelayMinutes']
X = merged_data.drop(columns=['ArrDelayMinutes'])

In [58]:
train = merged_data[merged_data['date'] < '2025-06-01']
test = merged_data[merged_data['date'] >= '2025-06-01']

X_train = train.drop(columns=['ArrDelayMinutes'])
y_train = train['ArrDelayMinutes']

X_test = test.drop(columns=['ArrDelayMinutes'])
y_test = test['ArrDelayMinutes']

In [62]:
X_train = X_train.dropna()
y_train = y_train.loc[X_train.index]  # keep target aligned
X_test = X_test.dropna()
y_test = y_test.loc[X_test.index]

Categorical columns: ['Reporting_Airline', 'Origin', 'Dest']
Numerical columns: ['Quarter', 'CRSArrTime', 'Distance', 'YEAR', 'tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun', 'delay_flag', 'DelayCategory', 'hour', 'airline_avg_delay']


C:\Users\Admin\AppData\Roaming\Python\Python312\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['tavg' 'tmin' 'tmax' 'prcp' 'snow' 'wdir' 'wspd' 'wpgt' 'pres' 'tsun']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


ValueError: Found array with 0 sample(s) (shape=(0, 18)) while a minimum of 1 is required by SimpleImputer.

In [65]:
# ===============================
# 1️⃣ Imports
# ===============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer


# ===============================
# 2️⃣ Safety Checks
# ===============================

# Ensure date is datetime
merged_data['date'] = pd.to_datetime(merged_data['date'], errors='coerce')

# Drop rows where target is missing
merged_data = merged_data.dropna(subset=['ArrDelayMinutes'])

# Sort by date (important for time-based modeling)
merged_data = merged_data.sort_values('date')


# ===============================
# 3️⃣ Feature Engineering
# ===============================

df = merged_data.copy()

# Extract date features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['weekday'] = df['date'].dt.weekday

# Convert CRSDepTime (e.g. 1330 -> 13)
if 'CRSDepTime' in df.columns:
    df['hour'] = (df['CRSDepTime'] // 100).astype(float)

# Drop raw date + CRSDepTime
df = df.drop(columns=['date', 'CRSDepTime'], errors='ignore')


# ===============================
# 4️⃣ Define X and y
# ===============================

X = df.drop(columns=['ArrDelayMinutes'])
y = df['ArrDelayMinutes']


# ===============================
# 5️⃣ Proper Time-Based Split (Safe)
# ===============================

# 80% train, 20% test — no empty set possible
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


# ===============================
# 6️⃣ Identify Column Types
# ===============================

categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()


# ===============================
# 7️⃣ Preprocessing Pipelines
# ===============================

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)


# ===============================
# 8️⃣ Full Modeling Pipeline
# ===============================

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    ))
])


# ===============================
# 9️⃣ Train
# ===============================

model.fit(X_train, y_train)


# ===============================
# 🔟 Predict
# ===============================

y_pred = model.predict(X_test)


# ===============================
# 1️⃣1️⃣ Evaluation
# ===============================

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n===== MODEL PERFORMANCE =====")
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")

Train shape: (16000, 25)
Test shape: (4000, 25)


C:\Users\Admin\AppData\Roaming\Python\Python312\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['tavg' 'tmin' 'tmax' 'prcp' 'snow' 'wdir' 'wspd' 'wpgt' 'pres' 'tsun']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(



===== MODEL PERFORMANCE =====
MAE  : 8.17
RMSE : 48.91
R²   : 0.0323


C:\Users\Admin\AppData\Roaming\Python\Python312\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['tavg' 'tmin' 'tmax' 'prcp' 'snow' 'wdir' 'wspd' 'wpgt' 'pres' 'tsun']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [66]:
# ===============================
# FEATURE IMPORTANCE
# ===============================

# Get feature names after preprocessing
feature_names = model.named_steps['preprocessor'].get_feature_names_out()

importances = model.named_steps['regressor'].feature_importances_

feature_importance_df = (
    pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    })
    .sort_values(by='importance', ascending=False)
)

print("\nTop 20 Important Features:")
print(feature_importance_df.head(20))


Top 20 Important Features:
                       feature  importance
5           num__DelayCategory    0.134987
4              num__delay_flag    0.130203
1              num__CRSArrTime    0.068217
2                num__Distance    0.053417
6                    num__hour    0.049674
40             cat__Origin_AVP    0.038740
7       num__airline_avg_delay    0.038615
175            cat__Origin_LAN    0.021118
224            cat__Origin_MTJ    0.020824
517              cat__Dest_MIA    0.019964
339              cat__Dest_AGS    0.014959
86             cat__Origin_COS    0.013686
8    cat__Reporting_Airline_AA    0.013365
538              cat__Dest_ORD    0.012407
623              cat__Dest_VCT    0.012220
43             cat__Origin_BDL    0.012216
621              cat__Dest_TYS    0.010010
261            cat__Origin_RDU    0.009047
231            cat__Origin_ORD    0.008996
190            cat__Origin_LIH    0.008701


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'regressor__n_estimators': [100, 200, 300],
    'regressor__max_depth': [None, 10, 20, 30],
    'regressor__min_samples_split': [2, 5, 10],
    'regressor__min_samples_leaf': [1, 2, 4]
}

search = RandomizedSearchCV(
    model,
    param_distributions=param_grid,
    n_iter=10,
    cv=3,
    scoring='neg_mean_absolute_error',
    verbose=1,
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)

print("\nBest Parameters:")
print(search.best_params_)

best_model = search.best_estimator_

y_pred = best_model.predict(X_test)

print("\n===== TUNED MODEL PERFORMANCE =====")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R²:", r2_score(y_test, y_pred))

Fitting 3 folds for each of 10 candidates, totalling 30 fits
